# 🩺 Vitalis — AI Healthcare Agent

Vitalis is a **rule-based reasoning agent** that helps a user go from "what am I feeling"
to "what should I do next." It is built entirely in Python so it can run offline inside
this notebook — no external API keys or network calls are required.

> ⚠️ **Disclaimer:** Vitalis is an educational, rule-based reasoning demo — **not a
> medical device** and it does not replace a licensed clinician. In a real emergency,
> always contact your local emergency services directly.

### What the agent does
| Module | What it does |
|---|---|
| 🧠 Symptom Intelligence | Parses a free-text symptom description, flags red-flag emergencies, and returns a severity + recommended next step |
| 🏥 Hospital Navigator | Ranks a local hospital directory by specialty fit, rating, and estimated travel time |
| 💊 Medicine & Interactions | Tracks medication reminders, adherence, and flags known drug-drug interactions |
| 👪 Family Risk Mapping | Builds a hereditary risk profile from a family health history |
| 🧪 Lab Report Explainer | Turns raw lab values into plain-language, flagged results |
| 📊 Dashboard | Summarizes everything the agent has tracked in one place |

### Project structure
```
AI-Agent-Project/
├── README.md
├── agent.ipynb          <- you are here
├── requirements.txt
├── data/
│   └── sample_data/     <- CSV knowledge bases the agent loads at runtime
└── screenshots/         <- example run screenshots
```


## 1. Setup

In [ ]:
import json
import re
import os
from dataclasses import dataclass, field, asdict
from datetime import datetime
from pathlib import Path
from typing import List, Dict, Optional

import pandas as pd

DATA_DIR = Path("data/sample_data")
STATE_PATH = Path("data/vitalis_state.json")

pd.set_option("display.max_colwidth", None)
print("Environment ready.")


## 2. Agent state

Vitalis keeps a small amount of state in memory while the notebook is running
(symptom-check history, medications, family profile) and persists it to
`data/vitalis_state.json` between sessions — mirroring the `localStorage`
persistence used in the original web prototype, but file-based so it works
identically in Jupyter or Colab.


In [ ]:
@dataclass
class VitalisState:
    triage_history: List[dict] = field(default_factory=list)
    medications: List[dict] = field(default_factory=list)
    family: List[dict] = field(default_factory=list)
    lab_checks: int = 0
    hospital_searches: int = 0

    def save(self):
        STATE_PATH.parent.mkdir(parents=True, exist_ok=True)
        STATE_PATH.write_text(json.dumps(asdict(self), indent=2))

    @classmethod
    def load(cls):
        if STATE_PATH.exists():
            data = json.loads(STATE_PATH.read_text())
            return cls(**data)
        return cls()

state = VitalisState.load()
print(f"Loaded state: {len(state.triage_history)} triage checks, "
      f"{len(state.medications)} medications, {len(state.family)} family members")


## 3. 🧠 Module 1 — Symptom Intelligence (triage reasoning engine)

Loads the symptom knowledge base and red-flag list from `data/sample_data/`, then scores
free-text symptom descriptions against them to produce a severity level, likely causes,
a recommended specialty, and a concrete next step.


In [ ]:
symptom_kb_df = pd.read_csv(DATA_DIR / "symptom_kb.csv")
red_flags_df = pd.read_csv(DATA_DIR / "red_flags.csv")

SYMPTOM_KB = [
    {"keywords": row["keywords"].split("|"), "specialty": row["specialty"],
     "weight": float(row["weight"]), "cause": row["likely_cause"]}
    for _, row in symptom_kb_df.iterrows()
]
RED_FLAGS = [{"kw": row["keyword"], "label": row["label"]} for _, row in red_flags_df.iterrows()]

SELF_HARM_FLAGS = [
    "suicide", "suicidal", "kill myself", "end my life",
    "want to die", "self harm", "self-harm", "hurt myself",
]

symptom_kb_df.head()


In [ ]:
def analyze_symptoms(text: str, age: Optional[int] = None, duration: str = "",
                      severity: str = "mild", history: str = "") -> dict:
    """Rule-based triage: mirrors the reasoning engine from the original prototype."""
    low = text.lower()

    self_harm_hit = any(f in low for f in SELF_HARM_FLAGS)
    red_hits = [r for r in RED_FLAGS if r["kw"] in low]
    matched = [k for k in SYMPTOM_KB if any(w in low for w in k["keywords"])]

    score = sum(m["weight"] for m in matched)
    score += len(red_hits) * 4
    score += {"mild": 0, "moderate": 1.5, "severe": 3}.get(severity, 0)

    d = duration.lower()
    if re.search(r"hour|today|since this morning|just now|minute", d):
        score += 1
    if re.search(r"week|month|year|chronic", d) and not red_hits:
        score -= 1

    if age is not None and (age < 5 or age > 75):
        score += 1

    h = history.lower()
    if re.search(r"pregnan|blood thinner|immunocompromised|chemotherapy|heart condition|diabet", h):
        score += 1

    if red_hits or score >= 6:
        sev_level = "emergency"
    elif score >= 3.5:
        sev_level = "urgent-care"
    elif score >= 1:
        sev_level = "clinic"
    else:
        sev_level = "self-care"

    spec_tally: Dict[str, float] = {}
    for m in matched:
        spec_tally[m["specialty"]] = spec_tally.get(m["specialty"], 0) + m["weight"]
    specialty = max(spec_tally, key=spec_tally.get) if spec_tally else "General Medicine"
    if sev_level == "emergency":
        specialty = "Emergency / ER"

    causes = list(dict.fromkeys(m["cause"] for m in matched))[:4]
    if not causes:
        causes = ["Non-specific symptom pattern — general evaluation recommended"]

    confidence = max(35, min(94, 40 + len(matched) * 9 + len(red_hits) * 10))

    summary_map = {
        "emergency": "This symptom pattern includes signs that need immediate medical attention.",
        "urgent-care": "This looks like it needs prompt medical attention, ideally today.",
        "clinic": "This is worth getting checked by a doctor, though it doesn't appear to be an emergency.",
        "self-care": "This looks manageable with self-care and monitoring for now.",
    }
    next_step_map = {
        "emergency": "Go to the nearest Emergency Room now, or call your local emergency number.",
        "urgent-care": "Visit an urgent care clinic or see a doctor within the next few hours.",
        "clinic": "Book a clinic appointment in the next 1-3 days if symptoms don't improve.",
        "self-care": "Rest, hydrate, and monitor. See a doctor if symptoms worsen or persist beyond a few days.",
    }

    explanation = (
        f"Vitalis matched patterns consistent with: {', '.join(causes).lower()}."
        if matched else
        "Vitalis didn't match a strong pattern from the description alone — "
        "when in doubt, a clinical opinion is the safest path."
    )
    if red_hits:
        explanation += f" Critical warning signs detected: {', '.join(r['label'] for r in red_hits)}."

    return {
        "severity": sev_level,
        "summary": summary_map[sev_level],
        "explanation": explanation,
        "likely_causes": causes,
        "recommended_specialty": specialty,
        "red_flags": [r["label"] for r in red_hits],
        "next_step": next_step_map[sev_level],
        "confidence": confidence,
        "self_harm_hit": self_harm_hit,
    }


def run_triage(symptoms: str, age: Optional[int] = None, duration: str = "",
                severity: str = "mild", history: str = "") -> dict:
    """Runs triage and logs the result into agent state (like clicking 'Analyze symptoms')."""
    result = analyze_symptoms(symptoms, age, duration, severity, history)

    if result["self_harm_hit"]:
        print("💛 You matter, and support is available right now. In the US you can call or "
              "text 988 (Suicide & Crisis Lifeline), available 24/7. Outside the US, search "
              "'[your country] suicide crisis helpline' or contact local emergency services.")

    entry = {**result, "symptoms": symptoms, "timestamp": datetime.now().isoformat()}
    state.triage_history.insert(0, entry)
    state.triage_history = state.triage_history[:8]
    state.save()
    return result


In [ ]:
# Demo: a routine symptom
demo1 = run_triage("I have a sore throat and a runny nose since yesterday", age=29, duration="1 day", severity="mild")
demo1


In [ ]:
# Demo: a red-flag emergency symptom
demo2 = run_triage("Sudden crushing chest pain and shortness of breath", age=54, duration="just now", severity="severe")
demo2


## 4. 🏥 Module 2 — Smart Hospital Navigator

Ranks a local (offline) hospital directory by specialty fit, rating, and estimated
travel time. In the original prototype, "Directions" opened a real Google Maps link —
we reproduce that same URL-building logic here.


In [ ]:
hospitals_df = pd.read_csv(DATA_DIR / "hospitals.csv")
HOSPITAL_POOL = [
    {"name": row["name"], "specialties": row["specialties"].split("|"),
     "rating": row["rating"], "base_eta": int(row["base_eta_min"])}
    for _, row in hospitals_df.iterrows()
]
hospitals_df


In [ ]:
import urllib.parse

def find_hospitals(specialty: str, city: str = "your area", top_n: int = 5) -> pd.DataFrame:
    state.hospital_searches += 1
    state.save()

    ranked = []
    for h in HOSPITAL_POOL:
        if specialty in h["specialties"]:
            fit = 2
        elif "General Medicine" in h["specialties"]:
            fit = 1
        else:
            fit = 0
        if fit > 0:
            ranked.append({**h, "fit": fit})

    ranked.sort(key=lambda h: (-h["fit"], h["base_eta"]))
    ranked = ranked[:top_n] or [dict(h, fit=0) for h in HOSPITAL_POOL[:top_n]]

    rows = []
    for i, h in enumerate(ranked, 1):
        query = urllib.parse.quote(f"{h['name']} {city}")
        maps_url = f"https://www.google.com/maps/search/?api=1&query={query}"
        rows.append({
            "rank": i, "hospital": h["name"], "rating": h["rating"],
            "eta_min": h["base_eta"], "specialties": ", ".join(h["specialties"]),
            "directions": maps_url,
        })
    return pd.DataFrame(rows)

find_hospitals("Cardiology", city="Mumbai")


## 5. 💊 Module 3 — Medicine Reminders & Interaction Checker

Tracks medications with dosage/schedule, logs adherence ("doses taken"), and
cross-checks every pair of active medications against a known drug-interaction table.


In [ ]:
interactions_df = pd.read_csv(DATA_DIR / "drug_interactions.csv")
INTERACTIONS = interactions_df.to_dict("records")
interactions_df


In [ ]:
def add_medication(name: str, dose: str = "", times: str = "", notes: str = "") -> dict:
    med = {"id": len(state.medications) + 1, "name": name, "dose": dose,
           "times": times, "notes": notes, "taken_log": []}
    state.medications.insert(0, med)
    state.save()
    return med


def mark_taken(med_id: int):
    for m in state.medications:
        if m["id"] == med_id:
            m["taken_log"].append(datetime.now().isoformat())
            state.save()
            return m
    raise ValueError(f"No medication with id {med_id}")


def check_interactions() -> pd.DataFrame:
    meds = [m["name"].lower() for m in state.medications]
    hits = []
    for i in range(len(meds)):
        for j in range(i + 1, len(meds)):
            for rule in INTERACTIONS:
                a, b = rule["drug_a"], rule["drug_b"]
                pair_match = (a in meds[i] and b in meds[j]) or (b in meds[i] and a in meds[j])
                if pair_match:
                    hits.append({
                        "pair": f"{state.medications[i]['name']} + {state.medications[j]['name']}",
                        "risk": rule["risk"], "note": rule["note"],
                    })
    if not hits:
        return pd.DataFrame([{"pair": "—", "risk": "none",
                               "note": "✅ No known interactions found among current medications."}])
    return pd.DataFrame(hits)


In [ ]:
# Demo
add_medication("Warfarin", dose="5mg", times="1x daily (evening)")
add_medication("Ibuprofen", dose="200mg", times="as needed", notes="for headaches")
check_interactions()


## 6. 👪 Module 4 — Family Health Risk Mapping

Builds a hereditary-risk profile from a family member list. Closer relatives
(parent / sibling / child) are weighted more heavily than a grandparent, matching
the original prototype's `RELATION_WEIGHT` table.


In [ ]:
family_risk_df = pd.read_csv(DATA_DIR / "family_risk_kb.csv")
RISK_KB = {row["condition"]: {"screenings": row["screenings"], "tips": row["tips"]}
           for _, row in family_risk_df.iterrows()}
RELATION_WEIGHT = {"Parent": 2, "Sibling": 2, "Child": 2, "Grandparent": 1}
family_risk_df


In [ ]:
def add_family_member(name: str, relation: str, age: Optional[int] = None, conditions: str = ""):
    member = {"id": len(state.family) + 1, "name": name, "relation": relation,
              "age": age, "conditions": conditions}
    state.family.insert(0, member)
    state.save()
    return member


def analyze_family_risk() -> pd.DataFrame:
    if not state.family:
        return pd.DataFrame([{"condition": "—", "level": "—", "reasoning": "Add a family member first."}])

    tally: Dict[str, float] = {}
    for f in state.family:
        w = RELATION_WEIGHT.get(f["relation"], 1)
        for cond in (f["conditions"] or "").split(","):
            cond = cond.strip().lower()
            if not cond:
                continue
            kb_key = next((k for k in RISK_KB if k in cond), None)
            if kb_key:
                tally[kb_key] = tally.get(kb_key, 0) + w

    if not tally:
        return pd.DataFrame([{"condition": "—", "level": "—",
                               "reasoning": "No hereditary-pattern conditions recognized. "
                                            "Try specific names like 'Type 2 diabetes' or 'hypertension'."}])

    rows = []
    for cond, pts in tally.items():
        level = "elevated" if pts >= 3 else "moderate" if pts >= 1.5 else "low"
        kb = RISK_KB[cond]
        rows.append({
            "condition": cond.title(), "level": level,
            "reasoning": f"{pts} relative-weighted point(s) logged for {cond}.",
            "recommended_screening": kb["screenings"], "prevention_tip": kb["tips"],
        })
    order = {"elevated": 0, "moderate": 1, "low": 2}
    rows.sort(key=lambda r: order[r["level"]])
    return pd.DataFrame(rows)


In [ ]:
# Demo
add_family_member("Dad", "Parent", age=61, conditions="Type 2 diabetes, hypertension")
add_family_member("Grandma", "Grandparent", age=84, conditions="heart disease")
analyze_family_risk()


## 7. 🧪 Module 5 — Lab Report & Prescription Explainer

Parses free-text lab values (one per line, e.g. `Glucose: 110 mg/dL`) and flags
anything outside the normal adult reference range, in plain language.


In [ ]:
lab_ref_df = pd.read_csv(DATA_DIR / "lab_reference.csv")
LAB_KB = [{"key": row["key"], "unit": row["unit"], "low": float(row["low"]),
           "high": float(row["high"]), "meaning": row["meaning"]}
          for _, row in lab_ref_df.iterrows()]
lab_ref_df


In [ ]:
NUM_RE = re.compile(r"([A-Za-z0-9 /%]+?)[:\-]?\s*([\d.]+)")

def explain_labs(text: str) -> pd.DataFrame:
    state.lab_checks += 1
    state.save()

    rows = []
    for line in (l.strip() for l in text.splitlines() if l.strip()):
        low_line = line.lower()

        bp_match = re.search(r"blood pressure[^\d]*(\d{2,3})\s*/\s*(\d{2,3})", low_line)
        if bp_match:
            sys_, dia_ = int(bp_match.group(1)), int(bp_match.group(2))
            status = "high" if (sys_ >= 140 or dia_ >= 90) else \
                     "elevated" if (sys_ >= 130 or dia_ >= 85) else "normal"
            rows.append({"label": "Blood pressure", "value": f"{sys_}/{dia_} mmHg",
                         "meaning": "arterial blood pressure", "range": "<120/80 normal",
                         "status": status})
            continue

        m = NUM_RE.match(line)
        if not m:
            continue
        term, val = m.group(1).strip().lower(), float(m.group(2))
        kb = next((k for k in LAB_KB if k["key"] in term), None)
        if not kb:
            continue
        status = "low" if val < kb["low"] else "high" if val > kb["high"] else "normal"
        rows.append({"label": m.group(1).strip(), "value": f"{val} {kb['unit']}",
                     "meaning": kb["meaning"], "range": f"{kb['low']}–{kb['high']} {kb['unit']}",
                     "status": status})

    if not rows:
        return pd.DataFrame([{"label": "—", "value": "—",
                               "meaning": "No recognized lab terms found. Try 'Glucose: 110 mg/dL' style lines.",
                               "range": "", "status": ""}])
    return pd.DataFrame(rows)


In [ ]:
# Demo
sample_report = """Glucose: 110 mg/dL
LDL: 145 mg/dL
Hemoglobin: 13.2 g/dL
Blood Pressure: 138/88"""

explain_labs(sample_report)


## 8. 📊 Module 6 — Health Dashboard

A snapshot of everything the agent has tracked this session, plus a simple
activity timeline — mirroring the "Health Dashboard" tab of the original prototype.


In [ ]:
def render_dashboard():
    total_doses = sum(len(m["taken_log"]) for m in state.medications)
    stats = pd.DataFrame([{
        "Symptom checks": len(state.triage_history),
        "Active medications": len(state.medications),
        "Doses logged": total_doses,
        "Family members tracked": len(state.family),
        "Hospital searches": state.hospital_searches,
        "Lab checks run": state.lab_checks,
    }])
    display(stats)

    events = []
    for h in state.triage_history:
        events.append((h["timestamp"], f"Symptom check — {h['severity'].replace('-', ' ')}"))
    for m in state.medications:
        for t in m["taken_log"]:
            events.append((t, f"Took {m['name']}"))
    events.sort(key=lambda e: e[0], reverse=True)

    if not events:
        print("No activity yet — try a symptom check or add a medication.")
    else:
        timeline = pd.DataFrame(events[:10], columns=["timestamp", "event"])
        display(timeline)

render_dashboard()


## 9. 🖥️ (Optional) Interactive CLI loop

Run the cell below in a live Jupyter/Colab session to interact with Vitalis
through simple text prompts instead of calling functions directly. Safe to skip —
this cell is optional and only runs when you execute it yourself.


In [ ]:
def interactive_menu():
    menu = """
Vitalis — choose an action:
  1) Symptom check
  2) Find a hospital
  3) Add a medication
  4) Check drug interactions
  5) Add a family member
  6) Analyze family risk
  7) Explain lab results
  8) Show dashboard
  0) Exit
"""
    while True:
        print(menu)
        choice = input("Choice: ").strip()
        if choice == "1":
            s = input("Describe your symptoms: ")
            display(pd.DataFrame([run_triage(s)]))
        elif choice == "2":
            sp = input("Specialty needed (e.g. Cardiology): ")
            city = input("City/area: ")
            display(find_hospitals(sp, city))
        elif choice == "3":
            n = input("Medicine name: "); d = input("Dosage: "); t = input("Times/day: ")
            add_medication(n, d, t)
            print("Added.")
        elif choice == "4":
            display(check_interactions())
        elif choice == "5":
            n = input("Name/relation label: "); r = input("Relation (Parent/Sibling/Child/Grandparent): ")
            c = input("Known conditions (comma separated): ")
            add_family_member(n, r, conditions=c)
            print("Added.")
        elif choice == "6":
            display(analyze_family_risk())
        elif choice == "7":
            print("Paste lab lines, blank line to finish:")
            lines = []
            while True:
                l = input()
                if not l:
                    break
                lines.append(l)
            display(explain_labs("\n".join(lines)))
        elif choice == "8":
            render_dashboard()
        elif choice == "0":
            print("Stay well. 👋")
            break
        else:
            print("Not a valid choice.")

# Uncomment to run interactively:
# interactive_menu()


---
### Notes
- All reasoning here is **rule-based** (keyword matching + weighted scoring), the same
  approach used in the original Vitalis prototype — deterministic, auditable, and fully
  offline, at the cost of being far less nuanced than a real clinician or a trained ML model.
- State persists to `data/vitalis_state.json` between runs of this notebook.
- To extend this into a true ML-backed agent, the `analyze_symptoms` / `explain_labs`
  functions are the natural place to swap in a trained classifier or an LLM call.
